
# FairWarn-SHS — Notebook 10A
## UCI Secondary-School External Validation

This notebook uses the **UCI Student Performance** dataset as an independent
secondary-school benchmark.

Important rules:

- The UCI data is **not merged** with the Ghana field data.
- The public dataset is evaluated as a separate external domain.
- No peer graph is invented for UCI.
- GraphSAGE is therefore not used in Notebook 10A.
- Logistic Regression, Random Forest, and a feature-only MLP are evaluated.
- Both full-feature and strict early-warning settings are tested.

Dataset citation:

Cortez, P. (2008). Student Performance [Dataset].
UCI Machine Learning Repository.
DOI: 10.24432/C5TG7T


In [ ]:
!pip -q install ucimlrepo pandas numpy scikit-learn matplotlib

In [ ]:

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    accuracy_score,
    brier_score_loss,
)

SEEDS = [42, 123, 456, 789, 1010]

PUBLIC_DATA_DIR = Path("data/public/uci_secondary")
OUTPUT_DIR = Path("outputs/public_validation/uci_secondary")

PUBLIC_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Folders ready.")



## Step 1 — Download the official UCI dataset

The notebook uses UCI dataset ID **320** through the official `ucimlrepo` package.

The public source is recorded in the metadata output for reproducibility.


In [ ]:

student_performance = fetch_ucirepo(id=320)

X = student_performance.data.features.copy()
y_source = student_performance.data.targets.copy()

print("Feature table shape:", X.shape)
print("Target table shape:", y_source.shape)
print()
print("Feature columns:")
print(X.columns.tolist())
print()
print("Target columns:")
print(y_source.columns.tolist())


In [ ]:

# Save a local snapshot of the downloaded public table.
uci_df = pd.concat([X, y_source], axis=1)

uci_df.to_csv(
    PUBLIC_DATA_DIR / "uci_student_performance_downloaded.csv",
    index=False
)

metadata = {
    "dataset_name": "UCI Student Performance",
    "uci_dataset_id": 320,
    "doi": "10.24432/C5TG7T",
    "repository": "UCI Machine Learning Repository",
    "purpose_in_fairwarn": "Independent secondary-school external validation",
    "merged_with_ghana_field_data": False,
}

with open(
    OUTPUT_DIR / "uci_source_metadata.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(metadata, file, indent=2)

print("Saved public-data snapshot and source metadata.")



## Step 2 — Define the external at-risk target

UCI's final grade is `G3`, measured from 0 to 20.

For this external validation, a student is labelled **at risk** when:

```text
G3 < 10
```

This is treated as a transparent operational threshold for this benchmark.
The original continuous `G3` value is then removed from the predictors.

`G1` and `G2` are earlier-period grades and are retained only in the
**full-feature** external model.

The **strict early-warning** version removes `G1` and `G2`.


In [ ]:

# Locate G3 safely.
if "G3" in uci_df.columns:
    final_grade = pd.to_numeric(uci_df["G3"], errors="raise")
else:
    raise KeyError(
        "G3 was not found in the UCI dataset returned by ucimlrepo."
    )

uci_df["TARGET_AtRisk"] = (final_grade < 10).astype(int)

print("Public records:", len(uci_df))
print("At-risk:", int(uci_df["TARGET_AtRisk"].sum()))
print("Not-at-risk:", int((uci_df["TARGET_AtRisk"] == 0).sum()))
print("At-risk rate:", round(uci_df["TARGET_AtRisk"].mean(), 4))


In [ ]:

# G3 is the target source and must never be used as a feature.
all_predictor_columns = [
    c for c in uci_df.columns
    if c not in {"G3", "TARGET_AtRisk"}
]

# Strict early-warning setting additionally removes the earlier grades.
strict_predictor_columns = [
    c for c in all_predictor_columns
    if c not in {"G1", "G2"}
]

print("Full-feature predictors:", len(all_predictor_columns))
print("Strict predictors:", len(strict_predictor_columns))
print()
print("Removed from strict setting:", ["G1", "G2"])


In [ ]:

def make_preprocessor(frame):
    numeric = frame.select_dtypes(
        include=[np.number]
    ).columns.tolist()

    categorical = [
        c for c in frame.columns
        if c not in numeric
    ]

    return ColumnTransformer([
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric,
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                (
                    "encoder",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                ),
            ]),
            categorical,
        ),
    ])


def calculate_metrics(
    y_true,
    probability,
    prediction,
):
    return {
        "AUC_ROC": roc_auc_score(
            y_true,
            probability,
        ),
        "AUC_PR": average_precision_score(
            y_true,
            probability,
        ),
        "Precision_AtRisk": precision_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "Recall_AtRisk": recall_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "F1_AtRisk": f1_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "Weighted_F1": f1_score(
            y_true,
            prediction,
            average="weighted",
            zero_division=0,
        ),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_true,
            prediction,
        ),
        "Accuracy": accuracy_score(
            y_true,
            prediction,
        ),
        "Brier_Score": brier_score_loss(
            y_true,
            probability,
        ),
    }


In [ ]:

def run_public_experiment(
    feature_columns,
    feature_setting,
):
    X_model = uci_df[
        feature_columns
    ].copy()

    y_model = uci_df[
        "TARGET_AtRisk"
    ].astype(int).to_numpy()

    result_rows = []
    prediction_rows = []

    for seed in SEEDS:
        row_indices = np.arange(
            len(uci_df)
        )

        (
            X_train,
            X_test,
            y_train,
            y_test,
            idx_train,
            idx_test,
        ) = train_test_split(
            X_model,
            y_model,
            row_indices,
            test_size=0.20,
            stratify=y_model,
            random_state=seed,
        )

        models = {
            "Logistic Regression": LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=seed,
            ),
            "Random Forest": RandomForestClassifier(
                n_estimators=500,
                class_weight="balanced",
                random_state=seed,
                n_jobs=-1,
            ),
            "Feature-only MLP": MLPClassifier(
                hidden_layer_sizes=(64, 32),
                alpha=0.0005,
                learning_rate_init=0.001,
                max_iter=500,
                early_stopping=True,
                random_state=seed,
            ),
        }

        for model_name, model in models.items():
            pipeline = Pipeline([
                (
                    "preprocess",
                    make_preprocessor(
                        X_train
                    ),
                ),
                ("model", model),
            ])

            pipeline.fit(
                X_train,
                y_train,
            )

            probability = pipeline.predict_proba(
                X_test
            )[:, 1]

            prediction = (
                probability >= 0.5
            ).astype(int)

            metrics = calculate_metrics(
                y_test,
                probability,
                prediction,
            )

            result_rows.append({
                "Dataset": "UCI Student Performance",
                "Feature_Setting": feature_setting,
                "Seed": seed,
                "Model": model_name,
                "Train_N": len(y_train),
                "Test_N": len(y_test),
                **metrics,
            })

            for (
                public_index,
                true_value,
                predicted_value,
                probability_value,
            ) in zip(
                idx_test,
                y_test,
                prediction,
                probability,
            ):
                prediction_rows.append({
                    "Dataset": "UCI Student Performance",
                    "Feature_Setting": feature_setting,
                    "Seed": seed,
                    "Model": model_name,
                    "Public_Row_Index": int(public_index),
                    "True_Label": int(true_value),
                    "Predicted_Label": int(predicted_value),
                    "AtRisk_Probability": float(probability_value),
                })

            print(
                f"{feature_setting:7s} | "
                f"{model_name:20s} | "
                f"seed {seed} | "
                f"AUC-PR={metrics['AUC_PR']:.4f} | "
                f"Recall={metrics['Recall_AtRisk']:.4f} | "
                f"F1={metrics['F1_AtRisk']:.4f}"
            )

    return (
        pd.DataFrame(result_rows),
        pd.DataFrame(prediction_rows),
    )


## Step 3 — Run full-feature public validation

In [ ]:

full_metrics_df, full_predictions_df = run_public_experiment(
    all_predictor_columns,
    "Full",
)


## Step 4 — Run strict early-warning public validation

In [ ]:

strict_metrics_df, strict_predictions_df = run_public_experiment(
    strict_predictor_columns,
    "Strict",
)


In [ ]:

all_metrics_df = pd.concat(
    [
        full_metrics_df,
        strict_metrics_df,
    ],
    ignore_index=True,
)

all_predictions_df = pd.concat(
    [
        full_predictions_df,
        strict_predictions_df,
    ],
    ignore_index=True,
)

summary_metrics = [
    "AUC_ROC",
    "AUC_PR",
    "Precision_AtRisk",
    "Recall_AtRisk",
    "F1_AtRisk",
    "Weighted_F1",
    "Balanced_Accuracy",
    "Accuracy",
    "Brier_Score",
]

summary_rows = []

for (
    feature_setting,
    model_name,
), group in all_metrics_df.groupby(
    [
        "Feature_Setting",
        "Model",
    ]
):
    row = {
        "Dataset": "UCI Student Performance",
        "Feature_Setting": feature_setting,
        "Model": model_name,
        "Seeds": group["Seed"].nunique(),
    }

    for metric in summary_metrics:
        row[
            f"{metric}_Mean"
        ] = group[metric].mean()

        row[
            f"{metric}_SD"
        ] = group[metric].std(
            ddof=1
        )

    summary_rows.append(row)

public_summary_df = pd.DataFrame(
    summary_rows
).sort_values(
    [
        "Feature_Setting",
        "AUC_PR_Mean",
    ],
    ascending=[
        True,
        False,
    ],
).reset_index(drop=True)

public_summary_df


In [ ]:

# Quantify how much performance changes when G1 and G2 are removed.
full_summary = (
    public_summary_df[
        public_summary_df[
            "Feature_Setting"
        ].eq("Full")
    ]
    .set_index("Model")
)

strict_summary = (
    public_summary_df[
        public_summary_df[
            "Feature_Setting"
        ].eq("Strict")
    ]
    .set_index("Model")
)

comparison_rows = []

for model_name in sorted(
    set(full_summary.index)
    & set(strict_summary.index)
):
    comparison_rows.append({
        "Model": model_name,
        "Full_AUC_PR": full_summary.loc[
            model_name,
            "AUC_PR_Mean",
        ],
        "Strict_AUC_PR": strict_summary.loc[
            model_name,
            "AUC_PR_Mean",
        ],
        "Delta_AUC_PR_Strict_Minus_Full": (
            strict_summary.loc[
                model_name,
                "AUC_PR_Mean",
            ]
            - full_summary.loc[
                model_name,
                "AUC_PR_Mean",
            ]
        ),
        "Full_Recall": full_summary.loc[
            model_name,
            "Recall_AtRisk_Mean",
        ],
        "Strict_Recall": strict_summary.loc[
            model_name,
            "Recall_AtRisk_Mean",
        ],
        "Delta_Recall_Strict_Minus_Full": (
            strict_summary.loc[
                model_name,
                "Recall_AtRisk_Mean",
            ]
            - full_summary.loc[
                model_name,
                "Recall_AtRisk_Mean",
            ]
        ),
        "Full_F1": full_summary.loc[
            model_name,
            "F1_AtRisk_Mean",
        ],
        "Strict_F1": strict_summary.loc[
            model_name,
            "F1_AtRisk_Mean",
        ],
        "Delta_F1_Strict_Minus_Full": (
            strict_summary.loc[
                model_name,
                "F1_AtRisk_Mean",
            ]
            - full_summary.loc[
                model_name,
                "F1_AtRisk_Mean",
            ]
        ),
    })

strict_vs_full_df = pd.DataFrame(
    comparison_rows
)

strict_vs_full_df


In [ ]:

plt.figure(figsize=(9, 5))

plot_df = public_summary_df.copy()

models = [
    "Logistic Regression",
    "Random Forest",
    "Feature-only MLP",
]

x = np.arange(len(models))
width = 0.35

for index, feature_setting in enumerate(
    ["Full", "Strict"]
):
    temp = (
        plot_df[
            plot_df[
                "Feature_Setting"
            ].eq(feature_setting)
        ]
        .set_index("Model")
        .reindex(models)
    )

    offset = (
        index - 0.5
    ) * width

    plt.bar(
        x + offset,
        temp["AUC_PR_Mean"],
        width,
        yerr=temp["AUC_PR_SD"],
        capsize=4,
        label=feature_setting,
    )

plt.xticks(
    x,
    models,
    rotation=20,
)
plt.ylabel("AUC-PR")
plt.title(
    "UCI external validation: full vs strict features"
)
plt.legend()
plt.tight_layout()
plt.show()



## What this notebook proves

Notebook 10A is an **external tabular robustness check**.

It does not prove that the Ghana model generalises perfectly to Portugal.
Instead, it tests whether similar student-risk modelling behaviour appears in a
different public secondary-school dataset.

GraphSAGE is intentionally excluded because the UCI dataset does not provide
student-to-student relational edges.

The graph-specific external validation is handled separately in Notebook 10B
using a dataset with genuine interaction records.


In [ ]:

public_summary_df.to_csv(
    OUTPUT_DIR / "fairwarn10a_uci_summary_mean_sd.csv",
    index=False,
)

all_metrics_df.to_csv(
    OUTPUT_DIR / "fairwarn10a_uci_metrics_by_seed.csv",
    index=False,
)

strict_vs_full_df.to_csv(
    OUTPUT_DIR / "fairwarn10a_uci_strict_vs_full.csv",
    index=False,
)

all_predictions_df.to_csv(
    OUTPUT_DIR / "fairwarn10a_uci_predictions.csv",
    index=False,
)

from google.colab import files

files.download(
    str(
        OUTPUT_DIR
        / "fairwarn10a_uci_summary_mean_sd.csv"
    )
)

files.download(
    str(
        OUTPUT_DIR
        / "fairwarn10a_uci_metrics_by_seed.csv"
    )
)

files.download(
    str(
        OUTPUT_DIR
        / "fairwarn10a_uci_strict_vs_full.csv"
    )
)

files.download(
    str(
        OUTPUT_DIR
        / "uci_source_metadata.json"
    )
)
